In [1]:
import pandas as pd
import sqlite3

# Load the Superstore Excel dataset
file_path = "../data/sample_-_superstore.xls"

df = pd.read_excel(file_path)

# Create a SQLite database in the project's data folder
conn = sqlite3.connect("../data/superstore.db")

# Load the dataset into a SQL table
df.to_sql("superstore", conn, if_exists="replace", index=False)

print("Database created successfully.")
print(f"Rows loaded: {len(df):,}")
print(f"Columns loaded: {len(df.columns)}")

Database created successfully.
Rows loaded: 10,194
Columns loaded: 21


In [3]:
query = """
SELECT *
FROM superstore
LIMIT 5;
"""

pd.read_sql_query(query, conn)

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country/Region,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,US-2023-103800,2023-01-03 00:00:00,2023-01-07 00:00:00,Standard Class,DP-13000,Darren Powers,Consumer,United States,Houston,...,77095,Central,OFF-PA-10000174,Office Supplies,Paper,"Message Book, Wirebound, Four 5 1/2"" X 4"" Form...",16.448,2,0.2,5.5512
1,2,US-2023-112326,2023-01-04 00:00:00,2023-01-08 00:00:00,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,...,60540,Central,OFF-BI-10004094,Office Supplies,Binders,GBC Standard Plastic Binding Systems Combs,3.540,2,0.8,-5.4870
2,3,US-2023-112326,2023-01-04 00:00:00,2023-01-08 00:00:00,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,...,60540,Central,OFF-LA-10003223,Office Supplies,Labels,Avery 508,11.784,3,0.2,4.2717
3,4,US-2023-112326,2023-01-04 00:00:00,2023-01-08 00:00:00,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,...,60540,Central,OFF-ST-10002743,Office Supplies,Storage,SAFCO Boltless Steel Shelving,272.736,3,0.2,-64.7748
4,5,US-2023-141817,2023-01-05 00:00:00,2023-01-12 00:00:00,Standard Class,MB-18085,Mick Brown,Consumer,United States,Philadelphia,...,19143,East,OFF-AR-10003478,Office Supplies,Art,Avery Hi-Liter EverBold Pen Style Fluorescent ...,19.536,3,0.2,4.8840


In [4]:
query = """
PRAGMA table_info(superstore);
"""

columns = pd.read_sql_query(query, conn)
columns[["name", "type"]]

,name,type
0,Row ID,INTEGER
1,Order ID,TEXT
2,Order Date,TIMESTAMP
3,Ship Date,TIMESTAMP
4,Ship Mode,TEXT
5,Customer ID,TEXT
6,Customer Name,TEXT
7,Segment,TEXT
8,Country/Region,TEXT
9,City,TEXT


In [5]:
query = """
SELECT
    ROUND(SUM(Sales), 2) AS total_sales,
    ROUND(SUM(Profit), 2) AS total_profit,
    SUM(Quantity) AS total_quantity,
    COUNT(DISTINCT "Order ID") AS total_orders,
    ROUND(SUM(Profit) / SUM(Sales) * 100, 2) AS profit_margin_pct
FROM superstore;
"""

pd.read_sql_query(query, conn)

,total_sales,total_profit,total_quantity,total_orders,profit_margin_pct
0,2326534.35,292296.81,38654,5111,12.56


In [6]:
query = """
SELECT
    Category,
    ROUND(SUM(Sales), 2) AS total_sales,
    ROUND(SUM(Profit), 2) AS total_profit,
    ROUND(SUM(Profit) / SUM(Sales) * 100, 2) AS profit_margin_pct
FROM superstore
GROUP BY Category
ORDER BY total_profit DESC;
"""

pd.read_sql_query(query, conn)

,Category,total_sales,total_profit,profit_margin_pct
0,Technology,839893.28,146543.38,17.45
1,Office Supplies,731893.31,126023.44,17.22
2,Furniture,754747.76,19730.00,2.61


In [7]:
query = """
SELECT
    "Sub-Category",
    ROUND(SUM(Sales), 2) AS total_sales,
    ROUND(SUM(Profit), 2) AS total_profit,
    ROUND(SUM(Profit) / SUM(Sales) * 100, 2) AS profit_margin_pct
FROM superstore
WHERE Category = 'Furniture'
GROUP BY "Sub-Category"
ORDER BY total_profit ASC;
"""

pd.read_sql_query(query, conn)

,Sub-Category,total_sales,total_profit,profit_margin_pct
0,Tables,208020.18,-17753.21,-8.53
1,Bookcases,115361.20,-3632.07,-3.15
2,Furnishings,95598.13,13891.74,14.53
3,Chairs,335768.25,27223.53,8.11


In [8]:
query = """
SELECT
    Discount,
    ROUND(SUM(Sales), 2) AS total_sales,
    ROUND(SUM(Profit), 2) AS total_profit,
    ROUND(AVG(Profit), 2) AS avg_profit
FROM superstore
GROUP BY Discount
ORDER BY Discount;
"""

pd.read_sql_query(query, conn)

,Discount,total_sales,total_profit,avg_profit
0,0.00,1105323.79,326718.59,66.34
1,0.10,54952.50,9099.97,94.79
2,0.15,27558.52,1418.99,27.29
3,0.20,773939.40,91079.95,24.58
4,0.30,104474.07,-10513.45,-45.71
5,0.32,14493.46,-2391.14,-88.56
6,0.40,116497.76,-23086.37,-111.53
7,0.45,5484.97,-2493.11,-226.65
8,0.50,58918.54,-20506.43,-310.70
9,0.60,7105.94,-6164.35,-41.37


In [9]:
query = """
SELECT
    "Sub-Category",
    ROUND(AVG(Discount) * 100, 2) AS avg_discount_pct,
    ROUND(SUM(Sales), 2) AS total_sales,
    ROUND(SUM(Profit), 2) AS total_profit,
    ROUND(SUM(Profit) / SUM(Sales) * 100, 2) AS profit_margin_pct
FROM superstore
WHERE Category = 'Furniture'
GROUP BY "Sub-Category"
ORDER BY total_profit ASC;
"""

pd.read_sql_query(query, conn)

,Sub-Category,avg_discount_pct,total_sales,total_profit,profit_margin_pct
0,Tables,25.81,208020.18,-17753.21,-8.53
1,Bookcases,21.53,115361.20,-3632.07,-3.15
2,Furnishings,13.81,95598.13,13891.74,14.53
3,Chairs,16.92,335768.25,27223.53,8.11


In [10]:
query = """
SELECT
    "Product Name",
    Category,
    "Sub-Category",
    ROUND(SUM(Sales), 2) AS total_sales,
    ROUND(SUM(Profit), 2) AS total_profit
FROM superstore
GROUP BY "Product Name", Category, "Sub-Category"
HAVING SUM(Profit) < 0
ORDER BY total_profit ASC
LIMIT 10;
"""

pd.read_sql_query(query, conn)

,Product Name,Category,Sub-Category,total_sales,total_profit
0,Cubify CubeX 3D Printer Double Head Print,Technology,Machines,11099.96,-8879.97
1,Lexmark MX611dhe Monochrome Laser Printer,Technology,Machines,16829.90,-4589.97
2,Cubify CubeX 3D Printer Triple Head Print,Technology,Machines,7999.98,-3839.99
3,Chromcraft Bull-Nose Wood Oval Conference Tabl...,Furniture,Tables,9917.64,-2876.12
4,Bush Advantage Collection Racetrack Conference...,Furniture,Tables,9544.72,-1934.40
5,GBC DocuBind P400 Electric Binding System,Office Supplies,Binders,17965.07,-1878.17
6,Cisco TelePresence System EX90 Videoconferenci...,Technology,Machines,22638.48,-1811.08
7,Martin Yale Chadless Opener Electric Letter Op...,Office Supplies,Supplies,16656.20,-1299.18
8,Balt Solid Wood Round Tables,Furniture,Tables,6518.75,-1201.06
9,BoxOffice By Design Rectangular and Half-Moon ...,Furniture,Tables,1706.25,-1148.44


In [11]:
query = """
SELECT
    Region,
    ROUND(SUM(Sales), 2) AS total_sales,
    ROUND(SUM(Profit), 2) AS total_profit,
    ROUND(SUM(Profit) / SUM(Sales) * 100, 2) AS profit_margin_pct
FROM superstore
GROUP BY Region
ORDER BY total_sales DESC;
"""

pd.read_sql_query(query, conn)

,Region,total_sales,total_profit,profit_margin_pct
0,West,739813.61,110798.82,14.98
1,East,691828.17,94883.26,13.71
2,Central,503170.67,39865.31,7.92
3,South,391721.91,46749.43,11.93
